# Fraud Detection — EDA & Model Selection

This notebook explores the Kaggle creditcard fraud dataset and compares candidate models before committing to the final architecture.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2. Load Data

In [ ]:
# Update path to your local copy of the dataset
df = pd.read_csv("../data/creditcard.csv")
print(df.shape)
df.head()

## 3. Class Imbalance Analysis

In [ ]:
fraud_rate = df["Class"].mean()
print(f"Fraud rate: {fraud_rate:.4%}")
df["Class"].value_counts().plot(kind="bar", title="Class Distribution")
plt.xlabel("Class (0=Legit, 1=Fraud)")
plt.ylabel("Count")
plt.show()

## 4. Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, ax in zip([0, 1], axes):
    df[df["Class"] == label]["Amount"].hist(bins=50, ax=ax)
    ax.set_title(f"Amount distribution — Class {label}")
    ax.set_xlabel("Amount")
plt.tight_layout()
plt.show()

## 5. Model Comparison

Compare Logistic Regression, Random Forest, and XGBoost on AUPRC (area under precision-recall curve) — the right metric for imbalanced datasets.

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=577, eval_metric="aucpr", random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "AUPRC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }

pd.DataFrame(results).T.sort_values("AUPRC", ascending=False)

## 6. Conclusion

TODO: Summarize which model was chosen and why, and note any preprocessing decisions that affect the production pipeline.